In [1]:
%pip install -q -U ultralytics huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 8.9 MB/s eta 0:00:00


In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
import gc
import json
import time
import cv2
import torch
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import torch.nn.functional as F

from scipy.optimize import linear_sum_assignment
from huggingface_hub import hf_hub_download
from ultralytics.models.sam import SAM3SemanticPredictor

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [5]:
DATASET_ROOT = Path("/content/datasets")

VAL_IMAGES_DIR = DATASET_ROOT / "images" / "val"
VAL_LABELS_DIR = DATASET_ROOT / "labels" / "val"

BLOCK_05_ROOT = Path(
    "/content/drive/MyDrive/"
    "vision_unit_02_outputs/"
    "block_05"
)

SAM3_OUTPUT_ROOT = BLOCK_05_ROOT / "sam3_demo"
SAM3_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = 0

print("GPU:", torch.cuda.get_device_name(0))
print( "Validation images:", VAL_IMAGES_DIR)

GPU: Tesla T4
Validation images: /content/datasets/images/val


**Download gated weight once**

In [ ]:
model = hf_hub_download(
    repo_id="facebook/sam3",
    filename="sam3.pt",
    token="HF_token"
)

print("Model:", model)

sam3.pt: reconstructing file:   0%|          |  0.00B / 3.45GB            

sam3.pt: downloading bytes:           |  0.00B            

Model: /root/.cache/huggingface/hub/models--facebook--sam3/snapshots/3c879f39826c281e95690f02c7821c4de09afae7/sam3.pt


**Initialize SAM3**

In [8]:
sam3_overrides = {
    "conf": 0.25,
    "task": "segment",
    "mode": "predict",
    "model" : model,
    "imgsz" : 1008,
    "quantize" : 16,
    "device" : DEVICE,
    "save": False,
    "verbose": False,
}

sam3_predictor = SAM3SemanticPredictor(overrides=sam3_overrides)

### Ground-truth preparation

**Polygon labels to instance masks**

In [9]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png",}

def find_validation_images():
    return sorted(
        path for path in VAL_IMAGES_DIR.iterdir()
        if (
            path.is_file() and path.suffix.lower() 
            in IMAGE_SUFFIXES
        )
    )

In [ ]:
def load_ground_truth_instances(image_path):
    image_bgr = cv2.imread(str(image_path))
    height, width = image_bgr.shape[:2]
    
    label_path = (
        VAL_LABELS_DIR / f"{image_path.stem}.txt"
    )

    instance_masks = []
    instance_boxes = []

    if not label_path.is_file():
        return (
            image_bgr, instance_masks, instance_boxes,
        )

    lines = label_path.read_text(encoding="utf-8").splitlines()

    for line in lines:
        values = line.split()

        if len(values) < 7:
            continue

        coordinates = np.asarray(values[1:], dtype=np.float32,
        ).reshape(-1, 2)

        pixel_points = (
            coordinates
            * np.asarray([width, height],dtype=np.float32)
        )

        pixel_points[:, 0] = np.clip(
            pixel_points[:, 0],
            0, width - 1,
        )

        pixel_points[:, 1] = np.clip(
            pixel_points[:, 1],
            0, height - 1,
        )

        pixel_points = np.rint(pixel_points).astype(np.int32)
        mask = np.zeros((height, width), dtype=np.uint8)

        cv2.fillPoly(mask, [pixel_points], 1)
        if mask.sum() == 0:
            continue

        ys, xs = np.where(mask > 0)

        box = [
            float(xs.min()), float(ys.min()),
            float(xs.max() + 1), float(ys.max() + 1),
        ]

        instance_masks.append(mask.astype(bool))
        instance_boxes.append(box)

    return (
        image_bgr,
        instance_masks,
        instance_boxes
    )